In [ ]:
#Step 1: Setup
from google.colab import drive
drive.mount('/content/drive')
!pip install timm -q

import os, sys, json, csv, random, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score
BASE = '/content/drive/MyDrive/capcapstone/dr_detection'
sys.path.insert(0, f'{BASE}/raw_data')
from model import HybridDRModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

stats = json.load(open(f'{BASE}/clean_data/dataset_stats.json'))
weights = torch.tensor([stats['class_weights'][str(i)] for i in range(5)],
                       dtype=torch.float32).to(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# test_loader = DataLoader(DRDataset('test'), batch_size=8, shuffle=False, num_workers=2)

In [ ]:
# Step 2: Ordinal loss
class OrdinalCELoss(nn.Module):
    """Class-weighted CE + lambda * (expected grade - true grade)^2."""
    def __init__(self, weights=None, lam=0.5, n=5):
        super().__init__()
        self.weights, self.lam, self.n = weights, lam, n
    def forward(self, logits, y):
        ce = nn.functional.cross_entropy(logits, y, weight=self.weights)
        probs = torch.softmax(logits.float(), dim=1)
        grades = torch.arange(self.n, device=logits.device, dtype=torch.float32)
        expected = (probs * grades).sum(1)
        return ce + self.lam * ((expected - y.float()) ** 2).mean()

In [ ]:
#Step 3: Train and validate function
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, L, Pr = [], [], []
    for x, y, *_ in loader:
        lg = model(x.to(device)).float()
        P += lg.argmax(1).cpu().tolist(); L += y.tolist()
        Pr += torch.softmax(lg, 1).cpu().tolist()
    P, L, Pr = np.array(P), np.array(L), np.array(Pr)
    try: auc = roc_auc_score(L, Pr, multi_class='ovr', average='macro')
    except ValueError: auc = float('nan')
    return dict(acc=accuracy_score(L, P), f1=f1_score(L, P, average='macro', zero_division=0),
                qwk=cohen_kappa_score(L, P, weights='quadratic'), auc=auc)

import gc
from torch.amp import autocast, GradScaler

def run(lam, seed=42, epochs=60, patience=10, lr=1e-4):
    seed_all(seed)
    model = HybridDRModel().to(device)
    crit = OrdinalCELoss(weights, lam)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
    scaler = GradScaler('cuda')
    best, bad, tag = -1, 0, f'lam{lam}_seed{seed}'
    ck = f'{BASE}/checkpoints/best_model_ordinal_{tag}.pth'
    log = f'{BASE}/outputs/ordinal_metrics_log_{tag}.csv'
    with open(log, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch','train_loss','val_acc','val_f1','val_qwk','val_auc'])
    for ep in range(1, epochs + 1):
        model.train(); tl = 0
        for x, y, *_ in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with autocast('cuda'):
                out = model(x)
            loss = crit(out.float(), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tl += loss.item()
        sched.step()
        with autocast('cuda'):
            m = evaluate(model, val_loader)
        print(f"{tag} ep{ep:02d} loss={tl/len(train_loader):.3f} val QWK={m['qwk']:.3f} F1={m['f1']:.3f}")
        with open(log, 'a', newline='') as f:
            csv.writer(f).writerow([ep, tl/len(train_loader), m['acc'], m['f1'], m['qwk'], m['auc']])
        if m['qwk'] > best:
            best, bad = m['qwk'], 0; torch.save(model.state_dict(), ck)
        else:
            bad += 1
            if bad >= patience: print('early stop'); break
    del model, opt, sched, scaler
    gc.collect(); torch.cuda.empty_cache()
    return ck, best

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as transforms

CLEAN = f'{BASE}/clean_data'
TRANSFORM = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class DRDataset(Dataset):
    def __init__(self, split):
        self.df = pd.read_csv(f'{CLEAN}/{split}/labels.csv')
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        p = f"{CLEAN}/{r['image_path']}"
        if not os.path.exists(p): p = f"{CLEAN}/{os.path.basename(os.path.dirname(CLEAN))}/{r['image_path']}"
        return TRANSFORM(Image.open(p).convert('RGB')), int(r['grade']), str(r['image_path'])

train_ds = DRDataset('train')

# balanced sampling, same idea as your Phase 2 loader
labels = train_ds.df['grade'].astype(int).values
cw = 1.0 / np.bincount(labels, minlength=5)
sampler = WeightedRandomSampler(cw[labels], num_samples=len(labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(DRDataset('val'),  batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(DRDataset('test'), batch_size=8, shuffle=False, num_workers=2)
print(len(train_ds), len(val_loader.dataset), len(test_loader.dataset))

330 41 42


In [ ]:
#Step 4: Run the experiments
os.makedirs(f'{BASE}/checkpoints', exist_ok=True)
results = {}
for lam in [0.0, 0.25, 0.5, 1.0]:
    results[lam] = run(lam)          # about 15-25 min each on T4
print(results)

lam0.0_seed42 ep01 loss=1.015 val QWK=0.507 F1=0.199
lam0.0_seed42 ep02 loss=0.602 val QWK=0.525 F1=0.375
lam0.0_seed42 ep03 loss=0.450 val QWK=0.813 F1=0.471
lam0.0_seed42 ep04 loss=0.264 val QWK=0.782 F1=0.519
lam0.0_seed42 ep05 loss=0.286 val QWK=0.789 F1=0.555
lam0.0_seed42 ep06 loss=0.268 val QWK=0.572 F1=0.509
lam0.0_seed42 ep07 loss=0.218 val QWK=0.560 F1=0.410
lam0.0_seed42 ep08 loss=0.174 val QWK=0.689 F1=0.415
lam0.0_seed42 ep09 loss=0.082 val QWK=0.648 F1=0.530
lam0.0_seed42 ep10 loss=0.119 val QWK=0.865 F1=0.496
lam0.0_seed42 ep11 loss=0.102 val QWK=0.822 F1=0.590
lam0.0_seed42 ep12 loss=0.104 val QWK=0.788 F1=0.504
lam0.0_seed42 ep13 loss=0.081 val QWK=0.673 F1=0.433
lam0.0_seed42 ep14 loss=0.049 val QWK=0.865 F1=0.487
lam0.0_seed42 ep15 loss=0.145 val QWK=0.781 F1=0.386
lam0.0_seed42 ep16 loss=0.093 val QWK=0.580 F1=0.403
lam0.0_seed42 ep17 loss=0.043 val QWK=0.792 F1=0.514
lam0.0_seed42 ep18 loss=0.052 val QWK=0.601 F1=0.403
lam0.0_seed42 ep19 loss=0.057 val QWK=0.803 F1

lam0.25_seed42 ep01 loss=1.286 val QWK=0.497 F1=0.272
lam0.25_seed42 ep02 loss=0.704 val QWK=0.600 F1=0.400
lam0.25_seed42 ep03 loss=0.527 val QWK=0.677 F1=0.363
lam0.25_seed42 ep04 loss=0.329 val QWK=0.563 F1=0.374
lam0.25_seed42 ep05 loss=0.387 val QWK=0.867 F1=0.605
lam0.25_seed42 ep06 loss=0.378 val QWK=0.535 F1=0.418
lam0.25_seed42 ep07 loss=0.257 val QWK=0.301 F1=0.338
lam0.25_seed42 ep08 loss=0.224 val QWK=0.631 F1=0.411
lam0.25_seed42 ep09 loss=0.109 val QWK=0.634 F1=0.405
lam0.25_seed42 ep10 loss=0.156 val QWK=0.803 F1=0.404
lam0.25_seed42 ep11 loss=0.130 val QWK=0.767 F1=0.576
lam0.25_seed42 ep12 loss=0.139 val QWK=0.750 F1=0.452
lam0.25_seed42 ep13 loss=0.112 val QWK=0.791 F1=0.396
lam0.25_seed42 ep14 loss=0.062 val QWK=0.730 F1=0.395
lam0.25_seed42 ep15 loss=0.188 val QWK=0.734 F1=0.403
early stop
lam0.5_seed42 ep01 loss=1.617 val QWK=0.575 F1=0.136
lam0.5_seed42 ep02 loss=0.829 val QWK=0.640 F1=0.406
lam0.5_seed42 ep03 loss=0.628 val QWK=0.706 F1=0.355
lam0.5_seed42 ep04 l

In [7]:
#Step 5: Pick by validation, test once
best_lam = max(results, key=lambda l: results[l][1])
print('Best lambda by VAL QWK:', best_lam, results[best_lam][1])

m = HybridDRModel().to(device)
m.load_state_dict(torch.load(results[best_lam][0], map_location=device))
val_m, test_m = evaluate(m, val_loader), evaluate(m, test_loader)
print('VAL ', val_m); print('TEST', test_m)

# append to the shared ablation table
row = dict(change=f'Ordinal loss (lambda={best_lam})', person='P4',
           **{f'val_{k}': round(v,4) for k,v in val_m.items()},
           **{f'test_{k}': round(v,4) for k,v in test_m.items()})
p = f'{BASE}/outputs/ablation_results.csv'
pd.DataFrame([row]).to_csv(p, mode='a', header=not os.path.exists(p), index=False)
print('✅ appended to ablation_results.csv')

Best lambda by VAL QWK: 0.5 0.889956208504026
VAL  {'acc': 0.6829268292682927, 'f1': 0.5409523809523809, 'qwk': np.float64(0.889956208504026), 'auc': np.float64(0.7977733377733378)}
TEST {'acc': 0.6428571428571429, 'f1': 0.49027777777777787, 'qwk': np.float64(0.750353806962921), 'auc': np.float64(0.878727247655819)}
✅ appended to ablation_results.csv


In [8]:
results = {0.5: run(0.5, epochs=2)}
print(results)

lam0.5_seed42 ep01 loss=1.617 val QWK=0.575 F1=0.136
lam0.5_seed42 ep02 loss=0.829 val QWK=0.640 F1=0.406
{0.5: ('/content/drive/MyDrive/capcapstone/dr_detection/checkpoints/best_model_ordinal_lam0.5_seed42.pth', np.float64(0.6396637371035537))}
